# 01 — Album & Artist ID Index

Builds and saves the master row-index files that all feature notebooks depend on.

**What it does:**
- Loads the full album and artist universes from the raw MusicBrainz parquet exports
- Saves `album_ids.pkl` and `artist_ids.pkl` to `data/features/`

**Why this exists:** Every sparse feature matrix uses these ID lists to guarantee row alignment — row `i` in `album_tags_matrix`, `album_labels_matrix`, `album_genre_matrix`, etc. all refer to the same album. Building the index once here, independently of any feature, means any feature notebook can be run (or re-run) without depending on another feature notebook having run first.

**Inputs:** `data/mb_album.parquet`, `data/mb_artist.parquet`

**Outputs to `data/features/`:** `album_ids.pkl`, `artist_ids.pkl`

**Run before:** all feature notebooks

## Imports

In [ ]:
import os
import pickle
import pandas as pd

## Load the Full Album & Artist Universes

The index is derived from `mb_album` and `mb_artist` rather than from any tag or label table. This is the key design decision: not every album has tags, and not every artist has tags. Deriving the index from tag data would silently drop untagged albums, leaving gaps in every downstream matrix. By anchoring to the full entity tables, every album and artist gets a guaranteed row — untagged entities simply have all-zero rows.

IDs are sorted so the index is deterministic across re-runs.

In [ ]:
unique_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)
unique_artist_ids = pd.Index(
    pd.read_parquet('../data/mb_artist.parquet', columns=['id'])['id'].sort_values()
)

print(f'Album universe : {len(unique_album_ids):,}')
print(f'Artist universe: {len(unique_artist_ids):,}')

## Save ID Mappings

Saved as plain Python lists inside pickle files. Downstream notebooks load these and call `pd.Index(album_ids).get_indexer(...)` or equivalent to map entity IDs to row positions.

In [ ]:
os.makedirs('../data/features', exist_ok=True)

with open('../data/features/album_ids.pkl', 'wb') as f:
    pickle.dump(unique_album_ids.tolist(), f)

with open('../data/features/artist_ids.pkl', 'wb') as f:
    pickle.dump(unique_artist_ids.tolist(), f)

print('Saved album_ids.pkl')
print('Saved artist_ids.pkl')